In [14]:
from openai import OpenAI
import os
from lxml import etree

token = ''
client = OpenAI(api_key=token)

import pymupdf
import os

def extract_text_from_pdf(pdf_path):
    document = pymupdf.open(pdf_path)
    text = ""
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()
    return text

def extract_text_from_tei(tei_path):

    tree = etree.parse(tei_path)
    root = tree.getroot()
    
    # Define TEI namespace
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}
    
    # Extract text from teiHeader (title, authors, etc.)
    title_el = root.xpath("//tei:titleStmt/tei:title", namespaces=ns)
    if title_el:
        title = " ".join(title_el[0].xpath(".//text()", namespaces=ns)).strip()
    else:
        title = "No title"
    
    # Extract abstract text
    abstract_paras = root.xpath("//tei:abstract//tei:p", namespaces=ns)
    abstract = " ".join(p.text for p in abstract_paras if p.text)
    
    # Extract body text (excluding anything in <back>)
    body_divs = root.xpath("//tei:body//tei:div", namespaces=ns)
    body_sections = []
    for div in body_divs:
        header = div.find("tei:head", namespaces=ns)
        section_title = " ".join(header.xpath(".//text()", namespaces=ns)).strip() if header is not None else ""
        paragraphs = div.findall(".//tei:p", namespaces=ns)
        paragraph_texts = [" ".join(p.xpath(".//text()", namespaces=ns)).strip() for p in paragraphs]
        section_block = f"{section_title}\n" + "\n".join(paragraph_texts) if section_title else "\n".join(paragraph_texts)
        body_sections.append(section_block)

    body = "\n".join(body_sections)
    
    # Combine all into a single text block
    output_text = f"Title: {title}\n\nAbstract: {abstract}\n\nPaper Body:\n{body}"

    # Print preview of the result
    return output_text  # Preview the first 1500 characters

def extract_titleabstract_from_tei(tei_path):

    tree = etree.parse(tei_path)
    root = tree.getroot()
    
    # Define TEI namespace
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}
    
    # Extract text from teiHeader (title, authors, etc.)
    title_el = root.xpath("//tei:titleStmt/tei:title", namespaces=ns)
    if title_el:
        title = " ".join(title_el[0].xpath(".//text()", namespaces=ns)).strip()
    else:
        title = "No title"
    
    # Extract abstract text
    abstract_paras = root.xpath("//tei:abstract//tei:p", namespaces=ns)
    abstract = " ".join(p.text for p in abstract_paras if p.text)
    
    # Combine all into a single text block
    output_text = f"Title: {title}\n\nAbstract: {abstract}"

    # Print preview of the result
    return output_text  # Preview the first 1500 characters
    
# Define the schema as JSON for validation
schema_json = {
    "name": "gene_information_extraction",
    "schema": {"gene_name": "<Gene name>",
               "species": "<Species name>",

                "properties": {
                "biological_processes": {
                "type": "array",
                "description": "List of biological processes or programs the gene is directly involved in. Use actual Gene Ontology Biological Process terms.",
                "items": {
                    "type": "string",
                    "sentence template": "[Gene] is involved in the [Biological Process].",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            },
            "molecular_functions": {
                "type": "array",
                "description": "List of molecular-level activities performed by the gene. Use actual Gene Ontology Molecular Function terms.",
                "items": {
                    "type": "string",
                    "sentence template": "[Gene] enables the [Molecular Function].",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            },
            "interactions": {
                "type": "array",
                "description": "List the genes that the gene physically interacts with.",
                "items": {
                    "type": "string",
                    "sentence template": "[Gene] interacts with [Gene/Protein].",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            },
            "pathways": {
                "type": "array",
                "description": "List the genetic pathways that the gene is part of. Use Gene Ontology terms if they exist.",
                "items": {
                    "type": "string",
                    "sentence template": "[Gene] is part of the [Pathway Name].",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            },
            "modifiers": {
                "type": "array",
                "description": "List genetic modifiers or chemical modifiers that either ameliorate or exacerbate specific disease phenotypes (e.g. the paralysis phenotype) or target the gene. Use the WormBase Phenotype Ontology terms if they exist.",
                "items": {
                    "type": "string",
                    "sentence template": "[Modifier] ameliorates or exacerbates the [disease  phenotype]",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            },
            "gene_expression": {
                "type": "array",
                "description": "List the tissue and sub-cellular expressions of the gene. For sub-cellular locations, use Gene Ontology Cellular Component terms. For tissue expression, use the WormBase Anatomy Ontology terms.",
                "items": {
                    "type": "string",
                    "sentence template": "[Gene] is expressed in [Tissue/Subcellular Location].",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            },
            "relevance_to_disease": {
                "type": "array",
                "description": "List those disease/s where the gene is directly implicated in the pathogenesis of disease or disease-like phenotypes. Use Disease Ontology terms. How has the study of this gene contributed to the understanding of the pathogenesis of disease",
                "items": {
                    "type": "string",
                    "sentence template": "[Gene] is used to study [Disease].",
                    "evidence_sentences": "<Evidence sentences from the paper.>"
                }
            }
        },
        "required": [
            "biological_processes",
            "molecular_functions",
            "interactions",
            "pathways",
            "modifiers",
            "gene_expression",
            "relevance_to_disease"
        ],
        "additionalProperties": False
    }

}

def create_prompt_from_text(text, schema_json, gene_list):
    instruction = f"""
    Extract structured information from the experimental results of the paper content provided below. Focus only on C. elegans genes ({gene_list}). The output should strictly adhere to the following JSON schema:

    {schema_json['schema']}
    
    Paper Content:
    {text}

    Make sure all fields are included. If a field has no information, set it as an empty array ([]). For each value, include all the actual sentences from the publication that directly support the statement. Ignore genes not related to *C. elegans*.
    """
    return instruction

def information_extraction(instruction, models):
    gpt_responses = {}
    for model_name in models:
        print(model_name)
        response = client.chat.completions.create(
            model= model_name,  # Or the specific model you have access to
            messages=[
                {"role": "system", "content": "You are an assistant that extracts structured information from scientific papers based on a strict schema."},
                {"role": "user", "content": f"{instruction}"}
            ],
            #temperature=0,  # Adjust the temperature for more/less creativity
        )
        response_content = response.choices[0].message.content.strip()
        if response_content.startswith("```json") and response_content.endswith("```"):
            json_content = response_content[7:-3].strip()
        else:
            json_content = response_content.strip()
        gpt_responses[model_name] = json_content
    
    return gpt_responses


48777
/Users/pengyuan/Downloads/download/PMID_33315870.tei
['aps-1', 'daf-1', 'daf-16', 'let-92', 'paa-1', 'pph-4.1', 'ptr-1', 'sur-6', 'pph-6', 'saps-1', 'pptr-1', 'ppfr-2', 'smk-1', 'let-9']
/Users/pengyuan/Downloads/download/PMID_12019228.tei
['eor-1', 'eor-2', 'ksr-1', 'lin-1', 'lin-2', 'lin-25', 'lin-45', 'sem-5', 'sur-2', 'sur-6', 'lin-4', 'let-3', 'let-34']
/Users/pengyuan/Downloads/download/PMID_30250464.tei
['abf-2', 'abf-3', 'age-1', 'daf-1', 'daf-16', 'dbl-1', 'lec-6', 'lec-8', 'lys-7', 'nsy-1', 'pmk-1', 'sek-1', 'skn-1', 'sod-3', 'spp-1', 'clec-8', 'clec-60', 'clec-6', 'clec-85']
/Users/pengyuan/Downloads/download/PMID_17183709.tei
['daf-1', 'daf-2', 'daf-16', 'elt-2', 'lec-6', 'clec-6', 'clec-67']
/Users/pengyuan/Downloads/download/PMID_16631156.tei
['bar-1', 'gsk-3', 'let-502', 'lin-1', 'lin-17', 'lin-44', 'mig-5', 'pop-1', 'pry-1', 'rho-1', 'wrm-1', 'lin-4', 'let-5', 'let-50']
/Users/pengyuan/Downloads/download/PMID_9210792.tei
['unc-8']
/Users/pengyuan/Downloads/downloa

JSONDecodeError: Expecting property name enclosed in double quotes: line 2 column 3 (char 4)

In [ ]:
import json
import os
import re

folder_path = ''

with open('GENE-DESCRIPTION-JSON_WB.json', 'r') as f:
    gene_data = json.load(f)

gene_list = [item['gene_name'].lower() for item in gene_data['data']]
print(len(gene_list))

models = ['o1']

pdf_files = os.listdir(folder_path)
count = 0

for pdf_file in pdf_files:
    if pdf_file.endswith('.tei'):
        
        pdf_path = os.path.join(folder_path, pdf_file)
        # print(pdf_path)
        print(pdf_path)      
        pdf_text = extract_titleabstract_from_tei(pdf_path)
        pdf_text = pdf_text.lower()
        
        targeted_genes = []
        for gene_name in gene_list:
            if gene_name in pdf_text:
                targeted_genes.append(gene_name)
        
        print(targeted_genes)
        if len(targeted_genes)>0:
            pdf_text = extract_text_from_tei(pdf_path)
            task_name = pdf_file[:-4]+'_structured_gpt_responses_genes_targeted.json'
            count += 1
            if not os.path.isfile(os.path.join(folder_path, task_name)):
                if len(pdf_text.split()) > 128000:
                    pdf_text = ' '.join(pdf_text.split()[:80000])
                print(len(pdf_text.split()))
                instruction = create_prompt_from_text(pdf_text, schema_json, ','.join(targeted_genes))
                gpt_response = information_extraction(instruction, models)
                responses = {}
                for model in models:
                    responses[model] = json.loads(gpt_response[model])
                task_name = pdf_file[:-4]+'_structured_gpt_responses_genes_targeted.json'
                with open(os.path.join(folder_path, task_name), 'w') as f:
                    json.dump(responses, f, indent=4)
print(count)

In [64]:
response

{'type': 'object',
 'properties': {'biological_processes': ['KVS-1 is involved in the biological process sensory perception of taste (GO:0050909).'],
  'molecular_functions': ['KVS-1 enables the molecular function voltage-gated potassium channel activity (GO:0005249).'],
  'interactions': [],
  'pathways': ['KVS-1 is part of the pathway potassium ion transmembrane transport (GO:0071805).'],
  'modifiers': ['C113S-KVS-1 ameliorates the neuronal apoptosis phenotype'],
  'gene_expression': ['KVS-1 is expressed in ASE neurons'],
  'relevance_to_disease': ["KVS-1 is used to study Alzheimer's disease (DOID:10652)."]},
 'required': ['biological_processes',
  'molecular_functions',
  'interactions',
  'pathways',
  'modifiers',
  'gene_expression',
  'relevance_to_disease'],
 'additionalProperties': False}